Faiss

Facebook AI Similarity Search(Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [10]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=2)
docs= text_splitter.split_documents(documents)
docs

[Document(metadata={'source': 'speech.txt'}, page_content="Humanity\n\nGood morning everyone,\n\nToday, I am here to speak about one of the most important qualities that makes us truly human — humanity.\n\nWe often think that being human simply means being born as a member of the human species. But being human and having humanity are not exactly the same thing. Humanity is about how we treat others, how we respond to someone's suffering, and how willing we are to help, forgive, understand, and care for people around us. It is about compassion, kindness, empathy, dignity, and respect.\n\nIn today's world, we have achieved things that previous generations could hardly imagine. We can communicate with someone thousands of kilometers away within seconds. We have developed powerful technologies, artificial intelligence, advanced medicines, modern transportation, and incredible scientific discoveries. We can explore space and access almost unlimited information through a small device in our 

In [12]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = FAISS.from_documents(docs, embeddings)
db

In [25]:
### querying
query = " who uses social media"
docs = db.similarity_search(query)
docs[0].page_content


'Therefore, we must remember that behind every screen is a real human being.\n\nWords written online can hurt just as deeply as words spoken face to face. Before insulting someone, spreading misinformation, or participating in online harassment, we should remember that there is a person on the other side of the screen.\n\nTechnology should make us more connected, not less compassionate.\n\nYoung people especially have an important role in building a more humane world.\n\nThe future belongs to us, but the future is not only about careers, salaries, technology, or economic growth. It is also about the kind of society we choose to create.\n\nWe should strive to become successful, but success should not come at the cost of our values.\n\nWe should become educated, but education should make us more understanding.\n\nWe should become powerful, but power should make us more responsible.\n\nWe should become wealthy, but wealth should give us opportunities to help others.'

As a Retriever

A retriever is a component in a RAG (Retrieval-Augmented Generation) system that finds the most relevant chunks of information from a collection of documents based on the user's question. After documents are split into chunks and converted into embeddings, the chunks are stored in a vector database. When the user asks a question, the retriever searches the stored data and returns the most relevant chunks, which are then given to the LLM to generate the final answer.


We can also convert the vectorestore into a Retriever class. This allows us to easily use it in other Langchain methods, which largely works with retrievers

In [26]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

'Therefore, we must remember that behind every screen is a real human being.\n\nWords written online can hurt just as deeply as words spoken face to face. Before insulting someone, spreading misinformation, or participating in online harassment, we should remember that there is a person on the other side of the screen.\n\nTechnology should make us more connected, not less compassionate.\n\nYoung people especially have an important role in building a more humane world.\n\nThe future belongs to us, but the future is not only about careers, salaries, technology, or economic growth. It is also about the kind of society we choose to create.\n\nWe should strive to become successful, but success should not come at the cost of our values.\n\nWe should become educated, but education should make us more understanding.\n\nWe should become powerful, but power should make us more responsible.\n\nWe should become wealthy, but wealth should give us opportunities to help others.'

Similarity serach wiht score

There are some FAISS specifc methods. One of them is similarity_search_with_score, which allows you to return not only the documnets but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [27]:
docs_and_sscore = db.similarity_search_with_score(query)
docs_and_sscore

[(Document(id='024a02bd-74e4-4a26-ae76-7977e0451fed', metadata={'source': 'speech.txt'}, page_content='Therefore, we must remember that behind every screen is a real human being.\n\nWords written online can hurt just as deeply as words spoken face to face. Before insulting someone, spreading misinformation, or participating in online harassment, we should remember that there is a person on the other side of the screen.\n\nTechnology should make us more connected, not less compassionate.\n\nYoung people especially have an important role in building a more humane world.\n\nThe future belongs to us, but the future is not only about careers, salaries, technology, or economic growth. It is also about the kind of society we choose to create.\n\nWe should strive to become successful, but success should not come at the cost of our values.\n\nWe should become educated, but education should make us more understanding.\n\nWe should become powerful, but power should make us more responsible.\n\nWe

We can directly pass vectors instead of sentences

In [28]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[-0.6361316442489624,
 0.6569454669952393,
 -3.7490367889404297,
 0.9756324887275696,
 1.437234878540039,
 -0.33103740215301514,
 0.6064585447311401,
 -1.1342895030975342,
 0.8190127611160278,
 0.1984522044658661,
 -0.35446715354919434,
 -0.8563219308853149,
 1.6939589977264404,
 1.544925332069397,
 -0.7158085107803345,
 -0.27834832668304443,
 -1.2902710437774658,
 0.026009362190961838,
 -1.3678617477416992,
 1.2139655351638794,
 -2.1823039054870605,
 -1.1380839347839355,
 -1.3212034702301025,
 0.13476938009262085,
 0.49300652742385864,
 -0.4285232722759247,
 0.27426648139953613,
 0.35263967514038086,
 -0.4922580420970917,
 -0.8731855154037476,
 0.1459960639476776,
 0.7258153557777405,
 -1.6909472942352295,
 -0.024772442877292633,
 0.8786117434501648,
 0.06062043458223343,
 -0.08778539299964905,
 -0.3296845257282257,
 0.2397831231355667,
 -0.05633874237537384,
 0.3054826259613037,
 0.9698206782341003,
 -0.6721315383911133,
 -0.8306066989898682,
 1.478177785873413,
 -1.1303842067718506,

In [30]:
docs_score = db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='024a02bd-74e4-4a26-ae76-7977e0451fed', metadata={'source': 'speech.txt'}, page_content='Therefore, we must remember that behind every screen is a real human being.\n\nWords written online can hurt just as deeply as words spoken face to face. Before insulting someone, spreading misinformation, or participating in online harassment, we should remember that there is a person on the other side of the screen.\n\nTechnology should make us more connected, not less compassionate.\n\nYoung people especially have an important role in building a more humane world.\n\nThe future belongs to us, but the future is not only about careers, salaries, technology, or economic growth. It is also about the kind of society we choose to create.\n\nWe should strive to become successful, but success should not come at the cost of our values.\n\nWe should become educated, but education should make us more understanding.\n\nWe should become powerful, but power should make us more responsible.\n\nWe 

In [31]:
### Saving and loading the FAISS index
db.save_local("faiss_index")

In [35]:
new_dfb = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
docs = new_dfb.similarity_search(query)
docs

[Document(id='024a02bd-74e4-4a26-ae76-7977e0451fed', metadata={'source': 'speech.txt'}, page_content='Therefore, we must remember that behind every screen is a real human being.\n\nWords written online can hurt just as deeply as words spoken face to face. Before insulting someone, spreading misinformation, or participating in online harassment, we should remember that there is a person on the other side of the screen.\n\nTechnology should make us more connected, not less compassionate.\n\nYoung people especially have an important role in building a more humane world.\n\nThe future belongs to us, but the future is not only about careers, salaries, technology, or economic growth. It is also about the kind of society we choose to create.\n\nWe should strive to become successful, but success should not come at the cost of our values.\n\nWe should become educated, but education should make us more understanding.\n\nWe should become powerful, but power should make us more responsible.\n\nWe 